# optim-agent quickstart

[Open this notebook in Colab](https://colab.research.google.com/github/Optim-Agent/optim-agent/blob/main/tutorials/quickstart.ipynb).

This walkthrough runs offline with the `mock` backend, so no model API key or agent CLI is required. It tunes a small service configuration, inspects the trial history, resumes from storage, and then shows how to switch to Claude Code, Codex, or OpenCode.

## 1. Import optim-agent

When running from a checkout, the local package is used. In a hosted notebook, the cell installs the PyPI package if needed.

In [ ]:
try:
    import optim_agent as oa
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optim-agent"])
    import optim_agent as oa

from pathlib import Path
import math

print("optim-agent", oa.__version__)

## 2. Define a measurable objective

The objective can be any Python function that returns one scalar. Here the scalar is a utility score for a toy service: higher accuracy helps, while latency and cost hurt.

In [ ]:
def evaluate_service(threshold, batch_size, cache_policy):
    cache_bonus = {"off": 0.0, "conservative": 0.025, "aggressive": 0.04}[cache_policy]
    cache_latency = {"off": 18.0, "conservative": 7.0, "aggressive": -6.0}[cache_policy]
    cache_cost = {"off": 0.0, "conservative": 0.12, "aggressive": 0.22}[cache_policy]

    accuracy = 0.70 + 0.22 * math.exp(-((threshold - 0.62) / 0.16) ** 2) + cache_bonus
    latency_ms = 38.0 + 1.15 * batch_size + cache_latency
    cost_per_1k = 0.18 + 0.004 * batch_size + cache_cost
    utility = accuracy - 0.0022 * latency_ms - 0.06 * cost_per_1k
    return {"accuracy": accuracy, "latency_ms": latency_ms, "cost_per_1k": cost_per_1k, "utility": utility}


def objective(trial):
    threshold = trial.suggest_float(
        "threshold", 0.20, 0.95,
        context="classification threshold; higher values trade recall for precision",
    )
    batch_size = trial.suggest_int(
        "batch_size", 1, 64, log=True,
        context="requests processed together; larger batches improve throughput but add latency",
    )
    cache_policy = trial.suggest_categorical(
        "cache_policy", ("off", "conservative", "aggressive"),
        context="response cache setting; aggressive is faster but more expensive",
    )
    return evaluate_service(threshold, batch_size, cache_policy)["utility"]

## 3. Run a study offline

`AgentSampler(backend="mock")` is deterministic and token-free. It is useful for checking that the objective, search space, storage, and result inspection all work before spending real agent calls.

In [ ]:
storage = Path("tutorial_study.json")
storage.unlink(missing_ok=True)

study = oa.create_study(
    direction="maximize",
    sampler=oa.AgentSampler(
        backend="mock",
        context="Tune a request-serving policy for high accuracy, low latency, and low cost.",
        history=5,
        explicit_reasoning=True,
        qualitative_notes=True,
        seed=7,
    ),
    storage=storage,
    seed=7,
)
study.optimize(objective, n_trials=12, verbose=False)

assert study.best_value is not None
print("best utility:", round(study.best_value, 4))
print("best params:", study.best_params)

## 4. Inspect what happened

A study keeps every trial, not just the best one. That makes benchmark claims and production tuning runs auditable.

In [ ]:
rows = []
for trial in study.trials:
    if trial.state != "complete":
        continue
    metrics = evaluate_service(**trial.params)
    rows.append({
        "trial": trial.number,
        "utility": trial.value,
        "accuracy": metrics["accuracy"],
        "latency_ms": metrics["latency_ms"],
        **trial.params,
    })

for row in sorted(rows, key=lambda item: item["utility"], reverse=True)[:5]:
    print(
        f"#{row['trial']:02d} utility={row['utility']:.4f} "
        f"accuracy={row['accuracy']:.3f} latency={row['latency_ms']:.1f}ms "
        f"params={{threshold={row['threshold']:.3f}, batch_size={row['batch_size']}, "
        f"cache_policy={row['cache_policy']}}}"
    )

## 5. Resume from storage

The same storage file can be reopened later. New trials continue from the saved history.

In [ ]:
resumed = oa.create_study(
    direction="maximize",
    sampler=oa.AgentSampler(backend="mock", seed=13),
    storage=storage,
    seed=13,
)
before = len(resumed.trials)
resumed.optimize(objective, n_trials=3, verbose=False)

assert before == 12
assert len(resumed.trials) == 15
print("trials before resume:", before)
print("trials after resume:", len(resumed.trials))
print("best after resume:", round(resumed.best_value, 4), resumed.best_params)

## 6. Switch to a real agent

After installing and authenticating Claude Code, Codex, or OpenCode locally, replace the sampler with one of these backends:

```python
sampler = oa.AgentSampler(
    backend="codex",  # or "claude" / "opencode"
    model="gpt-5.5",
    effort="medium",
    context="Tune a request-serving policy for high accuracy, low latency, and low cost.",
    history=5,
    explicit_reasoning=True,
    qualitative_notes=True,
)
```

Hosted notebooks usually do not have access to local CLI authentication, so the executable cells keep using `backend="mock"`.